In [1]:
import langchain
print("LangChain version:", langchain.__version__)

LangChain version: 1.2.15


In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="langchain")

In [3]:
import os
import json
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.callbacks import BaseCallbackHandler

In [4]:
# No API key required for local Ollama!
model = ChatOllama(
    model="qwen2.5-coder:7b"
)

prompt = ChatPromptTemplate.from_template(
    "Reply in exactly five words: LangChain makes development..."
)

# In modern LangChain, you can pass the prompt directly to the model 
# or use the pipe syntax for cleaner execution
chain = prompt | model

response = chain.invoke({})

print(response.content)

# messages = prompt.format_messages()
# response = model.invoke(messages)


# print(response.content)

LangChain simplifies language model integration.


In [5]:
# Define a sample input that looks like a real support ticket
ticket_text = """
Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Windows laptop.
This started after the update. Please fix ASAP.
Also: error code TIMEOUT_504 sometimes shows up.
Thanks, Pinal
"""
print(ticket_text)



Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Windows laptop.
This started after the update. Please fix ASAP.
Also: error code TIMEOUT_504 sometimes shows up.
Thanks, Pinal



In [6]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You convert support tickets into structured incident summaries."),
    ("human", """
Extract fields from this ticket and respond in JSON with these keys only:
severity (low|medium|high),
product_area,
issue_summary,
suspected_cause,
next_action,
tags (array of short strings)

Ticket:
{ticket}
""")
])

In [7]:
parser = JsonOutputParser()
# parser = CommaSeparatedListOutputParser()

In [8]:
class SimpleLogger(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print("\n--- Callback: Model Start ---")
        print("Messages being sent:")
        for m in messages[0]:
            print(f"[{m.type.upper()}] {m.content[:300]}")

    def on_chat_model_end(self, result, **kwargs):
        print("\n--- Callback: Model End ---")
        print("Model responded.")

In [ ]:
# # Build and run the chain imperatively (explicit: prompt -> model)
# messages = prompt.format_messages(ticket=ticket_text)
# response = model.invoke(messages, config={"callbacks": [SimpleLogger()]})

# raw_text = response.content

# print("\n--- Raw model output ---")
# print(raw_text)

chain = prompt | model | parser

response = chain.invoke(
    {"ticket": ticket_text},
    {"callbacks": [SimpleLogger()]}
    )

# raw_text = response.content

print("\n--- Raw model output ---")
# print(raw_text)
print(json.dumps(response, indent=4))



--- Callback: Model Start ---
Messages being sent:
[SYSTEM] You convert support tickets into structured incident summaries.
[HUMAN] 
Extract fields from this ticket and respond in JSON with these keys only:
severity (low|medium|high),
product_area,
issue_summary,
suspected_cause,
next_action,
tags (array of short strings)

Ticket:

Hi team - my dashboard is super slow since yesterday.
It takes 20-30 seconds to load. I am on Wind

--- Raw model output ---
{
    "severity": "high",
    "product_area": "Dashboard Performance",
    "issue_summary": "Slow dashboard loading and frequent TIMEOUT_504 errors since update",
    "suspected_cause": "Update introduced performance issues",
    "next_action": "Rollback update, investigate root cause, optimize code",
    "tags": [
        "performance",
        "update",
        "timeout"
    ]
}


In [15]:
# Parse the raw model output into a structured Python dictionary
structured = parser.parse(raw_text)
print("--- Parsed result ---")
print(type(structured))
print(json.dumps(structured, indent=4))

--- Parsed result ---
<class 'dict'>
{
    "severity": "high",
    "product_area": "Dashboard",
    "issue_summary": "Slow performance of dashboard with 20-30 second load time on Windows laptop since the update.",
    "suspected_cause": "Update-related issue causing performance degradation.",
    "next_action": "Fix and deploy update to resolve the performance issue.",
    "tags": [
        "dashboard",
        "performance",
        "update"
    ]
}
